
# Quantum Neural Network (VQC) Training
**SIH 2026 | Egreen Quanta | Heart Disease Detection**

This notebook trains a Variational Quantum Classifier (QNN) on Google Colab.
It also demonstrates **Continuous Learning** (fine-tuning on new hospital data).

## How to use this notebook:
1. Upload `heart_fhs_train.csv` and `heart_fhs_test.csv` to Colab using the sidebar.
2. Run all cells in order.
3. Download the trained weights (`heart_qnn_weights.json`) at the end.


In [ ]:

!pip install pennylane pennylane-lightning scikit-learn pandas numpy matplotlib seaborn -q


In [ ]:

import os
from google.colab import files

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Upload your CSV files
print("Upload heart_fhs_train.csv and heart_fhs_test.csv")
uploaded = files.upload()

for filename in uploaded:
    with open(f"data/{filename}", "wb") as f:
        f.write(uploaded[filename])
    print(f"Saved: data/{filename}")



## Training the Quantum Neural Network
The cell below contains the entire QNN architecture:
- **AngleEmbedding** encodes 6 patient vitals into 6 qubits
- **4 Variational Layers** of trainable Rot gates + CNOT ring entanglement
- **Parameter-Shift Rule** computes exact quantum gradients (no approximations)
- **Recall-Optimized Threshold** at -0.3 to aggressively catch sick patients
- **Continuous Learning** demonstrates fine-tuning on 50 new patients

**This will run for approximately 1 hour. Let it finish!**


In [ ]:
"""
Quantum Neural Network (VQC) Training Script â€” FINAL VERSION
=============================================================
Designed to run on Google Colab with lightning.qubit C++ backend.
Trains a Variational Quantum Classifier on the Heart Disease dataset.
Includes continuous_learning() for fine-tuning on new data.
"""
import sys
import time
import numpy as np
import pandas as pd
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os

# ============================================================
# CONFIG
# ============================================================
N_QUBITS = 6
N_LAYERS = 4
LEARNING_RATE = 0.01
EPOCHS = 12
BATCH_SIZE = 32
RANDOM_STATE = 42
RECALL_THRESHOLD = -0.3  # Lowered from 0 to maximize recall

np.random.seed(RANDOM_STATE)

# ============================================================
# 1. LOAD AND PREPARE DATA
# ============================================================
print("=" * 60)
print("QUANTUM NEURAL NETWORK (VQC) TRAINING")
print("=" * 60)

df_train = pd.read_csv("data/heart_fhs_train.csv")
df_test = pd.read_csv("data/heart_fhs_test.csv")

feature_names = [c for c in df_train.columns if c != "target"]
X_train = df_train[feature_names].values
y_train = df_train["target"].values
X_test = df_test[feature_names].values
y_test = df_test["target"].values

y_train_qnn = 2 * y_train - 1
y_test_qnn = 2 * y_test - 1

idx_0 = np.where(y_train == 0)[0]
idx_1 = np.where(y_train == 1)[0]
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

# OPTIMIZED for 1-hour run: 170 patients per class (340 total)
n_minority = 170
balanced_idx = np.concatenate([idx_0[:n_minority], idx_1[:n_minority]])
np.random.shuffle(balanced_idx)

X_balanced = X_train[balanced_idx]
y_balanced = y_train_qnn[balanced_idx]

print(f"Training set: {len(X_train)} total, {len(X_balanced)} balanced ({n_minority} per class)")
print(f"Test set: {len(X_test)} patients")
print(f"Features: {feature_names}")

# ============================================================
# 2. DEFINE THE QUANTUM NEURAL NETWORK (VQC)
# ============================================================
# Try lightning.qubit (C++ accelerated), fall back to default.qubit
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    print("Using lightning.qubit (C++ accelerated)")
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    print("WARNING: lightning.qubit not available, using default.qubit (slower)")

def variational_block(weights, wires):
    n_wires = len(wires)
    for i in range(n_wires):
        qml.Rot(weights[i, 0], weights[i, 1], weights[i, 2], wires=wires[i])
    for i in range(n_wires):
        qml.CNOT(wires=[wires[i], wires[(i + 1) % n_wires]])

@qml.qnode(dev, interface="autograd", diff_method="parameter-shift")
def qnn_circuit(weights, x):
    qml.AngleEmbedding(x, wires=range(N_QUBITS))
    for layer in range(N_LAYERS):
        variational_block(weights[layer], wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

def predict_single(weights, x, threshold=0.0):
    output = float(qnn_circuit(weights, x))
    return 1 if output >= threshold else -1

def predict_batch(weights, X, threshold=0.0):
    return np.array([predict_single(weights, x, threshold) for x in X])

# ============================================================
# 3. COST FUNCTION AND TRAINING LOOP
# ============================================================
def cost(weights, X_batch, y_batch):
    predictions = [qnn_circuit(weights, x) for x in X_batch]
    predictions = pnp.stack(predictions)
    return pnp.mean((predictions - y_batch) ** 2)

def train_qnn(X_data, y_data, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, initial_weights=None):

    X_data = pnp.array(X_data, requires_grad=False)
    y_data = pnp.array(y_data, requires_grad=False)

    if initial_weights is not None:
        weights = pnp.array(initial_weights, requires_grad=True)
        print("Fine-tuning from existing weights...")
    else:
        weights = pnp.array(
            np.random.uniform(0, 2 * np.pi, (N_LAYERS, N_QUBITS, 3)),
            requires_grad=True
        )
        print("Training from scratch...")

    opt = qml.GradientDescentOptimizer(stepsize=lr)
    n_samples = len(X_data)
    print(f"Dataset: {n_samples} samples | Epochs: {epochs} | Batch size: {batch_size}")
    print("-" * 60)

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(n_samples)
        X_shuffled = X_data[perm]
        y_shuffled = y_data[perm]

        epoch_cost = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            # FIX: Capture batch by value using default args to avoid closure bug
            Xb = X_shuffled[i:i+batch_size]
            yb = y_shuffled[i:i+batch_size]

            weights, batch_cost = opt.step_and_cost(
                lambda w, Xb=Xb, yb=yb: cost(w, Xb, yb),
                weights
            )
            epoch_cost += float(batch_cost)
            n_batches += 1

        avg_cost = epoch_cost / n_batches
        elapsed = time.time() - start_time

        # Show accuracy every 3 epochs (use threshold=0 during training for honest eval)
        if epoch % 3 == 0 or epoch == 1:
            preds = predict_batch(weights, X_data[:100], threshold=0.0)
            acc = accuracy_score(y_data[:100], preds)
            print(f"Epoch {epoch:2d}/{epochs} | Cost: {avg_cost:.4f} | Train Acc: {acc:.2%} | Time: {elapsed:.0f}s")
        else:
            print(f"Epoch {epoch:2d}/{epochs} | Cost: {avg_cost:.4f} | Time: {elapsed:.0f}s")

        history.append({"epoch": epoch, "cost": avg_cost})
        sys.stdout.flush()

    print(f"\nTraining complete in {time.time() - start_time:.0f}s")
    return weights, history

# ============================================================
# 4. CONTINUOUS LEARNING / FINE-TUNING FUNCTION
# ============================================================
def continuous_learning(existing_weights_path, new_X, new_y, epochs=5, lr=0.005):
    print("\n" + "=" * 60)
    print("CONTINUOUS LEARNING: Fine-tuning on new data")
    print("=" * 60)
    with open(existing_weights_path, "r") as f:
        saved = json.load(f)
    old_weights = np.array(saved["weights"])
    new_y_qnn = 2 * new_y - 1

    print(f"Loaded weights trained on {saved.get('n_training_samples', '?')} samples")
    print(f"Fine-tuning on {len(new_X)} new patients")

    updated_weights, history = train_qnn(new_X, new_y_qnn, epochs=epochs, lr=lr, initial_weights=old_weights)

    output_path = existing_weights_path.replace(".json", "_updated.json")
    save_weights(updated_weights, output_path, n_samples=saved.get("n_training_samples", 0) + len(new_X))
    return updated_weights

# ============================================================
# 5. SAVE / LOAD UTILITIES
# ============================================================
def save_weights(weights, path, n_samples=0):
    data = {
        "weights": weights.tolist(),
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "n_training_samples": n_samples,
        "recall_threshold": RECALL_THRESHOLD,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    size_bytes = os.path.getsize(path)
    n_params = N_LAYERS * N_QUBITS * 3
    print(f"Weights saved to {path} ({size_bytes} bytes, {n_params} trainable parameters)")
    print(f"  -> ZERO patient data stored. Fully privacy-preserving.")

# ============================================================
# 6. VISUALIZATION HELPERS
# ============================================================
def plot_training_curve(history):
    """Plot the cost curve to show the model learning over time."""
    epochs_list = [h["epoch"] for h in history]
    costs = [h["cost"] for h in history]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs_list, costs, 'b-o', linewidth=2, markersize=6)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Cost (MSE)", fontsize=12)
    ax.set_title("QNN Training Loss Curve", fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("qnn_training_curve.png", dpi=150)
    plt.show()
    print("Training curve saved to qnn_training_curve.png")

def plot_confusion_heatmap(y_true, y_pred, title_prefix="QNN"):
    """Generate side-by-side confusion matrix heatmaps."""
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Healthy', 'At Risk'],
                yticklabels=['Healthy', 'At Risk'], ax=axes[0],
                annot_kws={"size": 16})
    axes[0].set_title(f'{title_prefix} Confusion Matrix (Counts)', fontsize=13)
    axes[0].set_ylabel('Actual', fontsize=12)
    axes[0].set_xlabel('Predicted', fontsize=12)

    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Reds',
                xticklabels=['Healthy', 'At Risk'],
                yticklabels=['Healthy', 'At Risk'], ax=axes[1],
                annot_kws={"size": 16})
    axes[1].set_title(f'{title_prefix} Per-Class Accuracy (Recall)', fontsize=13)
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_xlabel('Predicted', fontsize=12)

    plt.tight_layout()
    fname = f"{title_prefix.lower().replace(' ', '_')}_confusion_matrix.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Heatmap saved to {fname}")

def print_metrics(y_true, y_pred, label=""):
    """Print all key metrics in a clean format."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{'=' * 40}")
    print(f"  {label} METRICS")
    print(f"{'=' * 40}")
    print(f"  Accuracy:   {acc:.4f}")
    print(f"  Precision:  {prec:.4f}  (Of flagged, how many truly sick)")
    print(f"  Recall:     {rec:.4f}  (Of all sick, how many caught)  <-- KEY")
    print(f"  F1-Score:   {f1:.4f}")
    print(f"{'=' * 40}")
    return acc, prec, rec, f1

# ============================================================
# 7. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":

    # ---- PHASE 1: TRAIN ----
    print("\n[PHASE 1] Training QNN on balanced dataset...")
    trained_weights, train_history = train_qnn(X_balanced, y_balanced)

    os.makedirs("models", exist_ok=True)
    save_weights(trained_weights, "models/heart_qnn_weights.json", n_samples=len(X_balanced))

    # Plot training loss curve
    plot_training_curve(train_history)

    # ---- PHASE 2: EVALUATE (with recall-maximizing threshold) ----
    print("\n[PHASE 2] Evaluating on unseen test set...")
    print(f"Using recall-maximizing threshold: {RECALL_THRESHOLD}")

    y_pred_qnn = predict_batch(trained_weights, X_test, threshold=RECALL_THRESHOLD)
    y_pred_01 = (y_pred_qnn + 1) // 2

    acc, prec, rec, f1 = print_metrics(y_test, y_pred_01, label="QNN (Recall-Optimized)")
    plot_confusion_heatmap(y_test, y_pred_01, title_prefix="QNN Recall-Optimized")

    # Also show what default threshold=0 would give (for comparison)
    print("\n[COMPARISON] With default threshold (0.0):")
    y_pred_default = predict_batch(trained_weights, X_test, threshold=0.0)
    y_pred_default_01 = (y_pred_default + 1) // 2
    print_metrics(y_test, y_pred_default_01, label="QNN (Default Threshold)")

    # ---- PHASE 3: CONTINUOUS LEARNING DEMO ----
    print("\n[PHASE 3] Demonstrating Continuous Learning (50 new patients)...")
    new_idx = np.random.choice(len(X_test), 50, replace=False)
    new_X, new_y = X_test[new_idx], y_test[new_idx]

    updated_weights = continuous_learning("models/heart_qnn_weights.json", new_X, new_y, epochs=5, lr=0.005)

    y_pred_updated = predict_batch(updated_weights, X_test, threshold=RECALL_THRESHOLD)
    y_pred_updated_01 = (y_pred_updated + 1) // 2

    print_metrics(y_test, y_pred_updated_01, label="QNN (After Continuous Learning)")
    plot_confusion_heatmap(y_test, y_pred_updated_01, title_prefix="QNN After Continuous Learning")

    print("\n" + "=" * 60)
    print("ALL DONE. Files saved:")
    print("  models/heart_qnn_weights.json         (initial weights)")
    print("  models/heart_qnn_weights_updated.json  (after continuous learning)")
    print("  qnn_training_curve.png                 (loss curve)")
    print("  qnn_confusion_matrix.png               (heatmap)")
    print("=" * 60)



## Download All Results
Run the cell below to download:
- **Trained weights** (JSON, zero patient data, fully privacy-preserving)
- **Training loss curve** (shows the model learning over epochs)
- **Confusion matrix heatmaps** (before and after continuous learning)


In [ ]:

from google.colab import files
import glob

# Download weights
for f in glob.glob("models/*.json"):
    files.download(f)
    print(f"Downloaded: {f}")

# Download all generated plots
for f in glob.glob("*.png"):
    files.download(f)
    print(f"Downloaded: {f}")

print("\nDone! Copy the JSON files into your project's models/ folder.")
